In [ ]:
!pip install -q transformers accelerate bitsandbytes
!pip install -q langchain langchain-community langchain-text-splitters langchain-core
!pip install -q pypdf sentence-transformers faiss-cpu
!pip install -q pandas openpyxl tabulate!pip install -q transformers accelerate bitsandbytes
!pip install -q langchain langchain-community langchain-text-splitters langchain-core
!pip install -q pypdf sentence-transformers faiss-cpu
!pip install -q pandas openpyxl tabulate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.5/331.5 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 21.3 MB/s eta 0:00:00
ERROR: Invalid requirement: 'tabulate!pip': Expected end or semicolon (after name and no valid version specifier)
    tabulate!pip
            ^


In [ ]:
import os
import json
import re
import torch
import pandas as pd
from typing import Dict, List, Optional
from dataclasses import dataclass, asdict
import warnings

warnings.filterwarnings('ignore')

from pypdf import PdfReader
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    pipeline
)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document


# ============================================================================
# 2. データ構造の定義
# ============================================================================
@dataclass
class FinancialData:
    年度: str
    営業収益: str = "取得失敗"
    税引前利益: str = "取得失敗"
    当期純利益: str = "取得失敗"
    総資産: str = "取得失敗"
    ROE: str = "取得失敗"

    def to_dict(self):
        return asdict(self)


# ============================================================================
# 3. 正確な数値抽出クラス
# ============================================================================
class AccurateNumericalExtractor:

    def _get_value_for_column(
        self,
        text_block: str,
        keyword: str,
        unit_regex: str,
        target_column: int
    ) -> Optional[str]:

        next_metric_keywords = [
            "営業収益", "税引前利益", "親会社の所有者に帰属する当期利益",
            "資産合計", "純資産合計", "親会社の所有者に帰属する持分当期利益率",
            "総資産経常利益率", "自己資本比率", "一株当たり当期純利益",
            "一株当たり純資産額", "従業員数"
        ]

        next_metric_pattern = '|'.join(
            [re.escape(k) for k in next_metric_keywords if k != keyword]
        )

        pattern = (
            rf'^{re.escape(keyword)}\s*{re.escape(unit_regex)}\s*'
            rf'(.*?)(?=\n|{next_metric_pattern}|\Z)'
        )

        match = re.search(pattern, text_block, re.MULTILINE)

        if not match:
            fallback = (
                rf'^{re.escape(keyword)}.*?(\s+.*?)?'
                rf'(?=\n|{next_metric_pattern}|\Z)'
            )
            match = re.search(fallback, text_block, re.MULTILINE)

        if match:
            values_str = match.group(1).strip() if match.group(1) else ""
            all_raw_values = re.findall(r'[\d,]+(?:\.\d+)?|－', values_str)
            numeric_values = [v for v in all_raw_values if v != '－']

            if len(numeric_values) >= target_column:
                return numeric_values[target_column - 1]

        return None

    def extract_2025_data(self, pdf_path: str) -> FinancialData:
        reader = PdfReader(pdf_path)
        page5_text = reader.pages[4].extract_text()

        data = FinancialData(年度="2025年3月期")
        column_index = 5  # 2025年3月期は右端列

        mapping = [
            ("営業収益", "営業収益", "（百万円）"),
            ("税引前利益", "税引前利益", "（百万円）"),
            ("親会社の所有者に帰属する当期利益", "当期純利益", "（百万円）"),
            ("資産合計", "総資産", "（百万円）"),
            ("親会社の所有者に帰属する持分当期利益率", "ROE", "（％）")
        ]

        for jp_keyword, attr_name, unit_str in mapping:
            value = self._get_value_for_column(page5_text, jp_keyword, unit_str, column_index)

            if value:
                if unit_str == "（％）":
                    setattr(data, attr_name, f"{value}%")
                else:
                    setattr(data, attr_name, f"{value} 百万円")

        return data


# ============================================================================
# 4. RAG & Llama 要約クラス
# ============================================================================
class HybridAnalyzer:

    def __init__(self, pdf_path: str):
        self.pdf_path = pdf_path
        self.full_text = ""
        self.numerical_data = None
        self.vectorstore = None
        self.summarizer_pipeline = None

    def prepare(self):
        print("1/4: 数値データを抽出中...")
        extractor = AccurateNumericalExtractor()
        self.numerical_data = extractor.extract_2025_data(self.pdf_path)

        print("2/4: 全文テキストを読み込み中（50ページ）...")
        reader = PdfReader(self.pdf_path)
        for i in range(min(50, len(reader.pages))):
            self.full_text += reader.pages[i].extract_text() + "\n"

        print("3/4: RAGデータベースを構築中...")
        embeddings = HuggingFaceEmbeddings(model_name="intfloat/multilingual-e5-small")
        splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=100)
        docs = [Document(page_content=t) for t in splitter.split_text(self.full_text)]
        self.vectorstore = FAISS.from_documents(docs, embeddings)

        print("4/4: Llama-3.2-3B モデルをロード中...")
        model_id = "meta-llama/Llama-3.2-3B-Instruct"

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16
        )

        tokenizer = AutoTokenizer.from_pretrained(model_id)
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            quantization_config=bnb_config,
            device_map="auto"
        )

        self.summarizer_pipeline = pipeline(
            "text-generation",
            model=model,
            tokenizer=tokenizer
        )

        print("--- 全準備完了 ---")

    def analyze(self, query: str):
        search_results = self.vectorstore.similarity_search(query, k=8)
        context = "\n\n".join([d.page_content for d in search_results])

        num_info = json.dumps(self.numerical_data.to_dict(), ensure_ascii=False, indent=2)

        prompt = f"""
あなたはプロの金融アナリストです。

【絶対ルール（最重要）】
- 【正確な数値データ】に存在しない数値を新しく生成してはいけない
- 増減額・割合・推定値・補足数値を追加してはいけない
- 数値は完全一致でのみ使用すること
- 不明な場合は「記載なし」と書くこと

【目的】
決算の事実のみを簡潔に整理する。

【正確な数値データ】
{num_info}

【参考文書】
{context}

【出力形式】
・業績ハイライト
・業績要因
・今後の見通し

箇条書きで出力：
"""

        outputs = self.summarizer_pipeline(
            [{"role": "user", "content": prompt}],
            max_new_tokens=800,
            temperature=0.2
        )

        return outputs[0]["generated_text"][-1]["content"]


# ============================================================================
# 5. 実行
# ============================================================================
def main():
    from google.colab import files

    print("トヨタ自動車の有価証券報告書（PDF）をアップロードしてください。")
    uploaded = files.upload()
    pdf_file = list(uploaded.keys())[0]

    analyzer = HybridAnalyzer(pdf_file)
    analyzer.prepare()

    question = "2025年3月期の業績ハイライトと、その主な要因、今後の見通しを教えてください。"
    result = analyzer.analyze(question)

    print("\n" + "=" * 30 + " 分析結果 " + "=" * 30)
    print(result)
    print("=" * 68)
    print("\n✅ すべて完了！")


if __name__ == "__main__":
    main()

トヨタ自動車の有価証券報告書（PDF）をアップロードしてください。


Saving トヨタ_2025_03_有価証券報告書.pdf to トヨタ_2025_03_有価証券報告書 (3).pdf
1/4: 数値データを抽出中...
2/4: 全文テキストを読み込み中（50ページ）...
3/4: RAGデータベースを構築中...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


4/4: Llama-3.2-3B モデルをロード中...


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- 全準備完了 ---

============================== 分析結果 ==============================
このデータは、2025年3月期のトヨタ自動車の決算データです。以下はその内容を簡潔に整理したものです。

**業績ハイライト**

* 榜表順の業績は次のとおりです。
 + 榜表順の営業収益：48兆367億円（前期比増減 2兆9,413億円、6.5%）
 + 榜表順の営業利益：４兆7,955億円（前期比増減 △5,573億円、 △10.4%）
 + 榜表順の税引前利益：６兆4,145億円（前期比増減 △5,504億円、 △7.9%）
 + 榜表順の当期純利益：４兆7,650億円（前期比増減 △1,798億円、 △3.6%）

**業績要因**

* 榜表順の業績の主な増減要因は、次のとおりです。
 + 榜表順の営業面の努力：1,450億円
 + 榜表順の原価改善の努力：±0億円
 + 榜表順の為替変動の影響：5,900億円
 + 榜表順の諸経費の増減・低減努力： △9,900億円
 + 榜表順のその他： △3,023億円

**今後の見通し**

* トヨタ自動車は、将来に向けて、モノづくりの変革を目指す「未来工場」のプロジェクトを立ち上げました。
* 自動化の大幅な拡充や多様な働き方の導入など、10年先、50年先を見据えて、生産性向上とやりがいにつながる踏み込んだ取り組みを検討しています。
* 生産年齢人口の減少と建屋・設備の老朽化が進むことによって、モノづくりの基盤を守り抜くことが難しくなります。

✅ すべて完了！


In [ ]:
!pip install evaluate bert-score sentence-transformers

import evaluate
from bert_score import score as bert_score
from sentence_transformers import SentenceTransformer, util

# ---------------------------------------------------------
# 1. ROUGE
# ---------------------------------------------------------
def calc_rouge(pred, ref):
    rouge = evaluate.load("rouge")
    result = rouge.compute(predictions=[pred], references=[ref])
    return result

# ---------------------------------------------------------
# 2. BLEU
# ---------------------------------------------------------
def calc_bleu(pred, ref):
    bleu = evaluate.load("bleu")
    result = bleu.compute(predictions=[pred], references=[ref])
    return result

# ---------------------------------------------------------
# 3. BERTScore（意味類似度）
# ---------------------------------------------------------
def calc_bertscore(pred, ref):
    P, R, F1 = bert_score([pred], [ref], lang="ja")
    return {
        "precision": float(P[0]),
        "recall": float(R[0]),
        "f1": float(F1[0])
    }

# ---------------------------------------------------------
# 4. Sentence-BERT 類似度（RAGに最適）
# ---------------------------------------------------------
model_sbert = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

def calc_sbert_similarity(pred, ref):
    emb_pred = model_sbert.encode(pred, convert_to_tensor=True)
    emb_ref = model_sbert.encode(ref, convert_to_tensor=True)
    sim = util.cos_sim(emb_pred, emb_ref).item()
    return sim

# ---------------------------------------------------------
# 5. まとめて評価する関数
# ---------------------------------------------------------
def evaluate_summary(pred, ref):
    return {
        "ROUGE": calc_rouge(pred, ref),
        "BLEU": calc_bleu(pred, ref),
        "BERTScore": calc_bertscore(pred, ref),
        "SBERT_similarity": calc_sbert_similarity(pred, ref)
    }

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 8.1 MB/s eta 0:00:00


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
!pip install rouge_score
pred = """このデータは、2025年3月期のトヨタ自動車の決算データです。以下はその内容を簡潔に整理したものです。

**業績ハイライト**

* 榜表順の業績は次のとおりです。
 + 榜表順の営業収益：48兆367億円（前期比増減 2兆9,413億円、6.5%）
 + 榜表順の営業利益：４兆7,955億円（前期比増減 △5,573億円、 △10.4%）
 + 榜表順の税引前利益：６兆4,145億円（前期比増減 △5,504億円、 △7.9%）
 + 榜表順の当期純利益：４兆7,650億円（前期比増減 △1,798億円、 △3.6%）

**業績要因**

* 榜表順の業績の主な増減要因は、次のとおりです。
 + 榜表順の営業面の努力：1,450億円
 + 榜表順の原価改善の努力：±0億円
 + 榜表順の為替変動の影響：5,900億円
 + 榜表順の諸経費の増減・低減努力： △9,900億円
 + 榜表順のその他： △3,023億円

**今後の見通し**

* トヨタ自動車は、将来に向けて、モノづくりの変革を目指す「未来工場」のプロジェクトを立ち上げました。
* 自動化の大幅な拡充や多様な働き方の導入など、10年先、50年先を見据えて、生産性向上とやりがいにつながる踏み込んだ取り組みを検討しています。
* 生産年齢人口の減少と建屋・設備の老朽化が進むことによって、モノづくりの基盤を守り抜くことが難しくなります。"""

ref = """【2025年3月期 数値事実】

営業収益：48兆367億円
営業利益：４兆7,955億円
税引前利益：6兆4,145億円
当期利益：4兆7,650億円

【業績要因（PDF記載）】
・競争激化
・為替変動の影響
・諸経費の増減"""

result = evaluate_summary(pred, ref)
result

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'ROUGE': {'rouge1': np.float64(0.47058823529411764),
  'rouge2': np.float64(0.2448979591836735),
  'rougeL': np.float64(0.392156862745098),
  'rougeLsum': np.float64(0.47058823529411764)},
 'BLEU': {'bleu': 0.0,
  'precisions': [0.0, 0.0, 0.0, 0.0],
  'brevity_penalty': 1.0,
  'length_ratio': 6.1,
  'translation_length': 61,
  'reference_length': 10},
 'BERTScore': {'precision': 0.6207358241081238,
  'recall': 0.7877988815307617,
  'f1': 0.6943598389625549},
 'SBERT_similarity': 0.7902313470840454}

In [ ]:
import re
import numpy as np

def extract_numbers(text):
    # カンマ付きの数値を抽出
    nums = re.findall(r'\d[\d,]*', text)
    # カンマを除去して整数に変換
    return [int(n.replace(",", "")) for n in nums]

def numerical_accuracy(pred, ref):
    pred_nums = extract_numbers(pred)
    ref_nums = extract_numbers(ref)

    if len(ref_nums) == 0:
        return {"error": "ref に数値がありません"}

    # 完全一致率
    exact_matches = sum(p == r for p, r in zip(pred_nums, ref_nums))
    exact_match_rate = exact_matches / len(ref_nums)

    # 相対誤差
    relative_errors = []
    for p, r in zip(pred_nums, ref_nums):
        if r != 0:
            relative_errors.append(abs(p - r) / r)

    avg_relative_error = np.mean(relative_errors) if relative_errors else None

    return {
        "pred_numbers": pred_nums,
        "ref_numbers": ref_nums,
        "exact_match_rate": exact_match_rate,
        "avg_relative_error": avg_relative_error
    }

In [ ]:
def financial_precision(pred, ref):

    pred_vals = set(extract_money(pred))
    ref_vals = set(extract_money(ref))

    if len(pred_vals) == 0:
        return 0.0

    correct = len(pred_vals & ref_vals)
    return correct / len(pred_vals)

In [ ]:
def financial_recall(pred, ref):

    pred_vals = set(extract_money(pred))
    ref_vals = set(extract_money(ref))

    if len(ref_vals) == 0:
        return 0.0

    correct = len(pred_vals & ref_vals)
    return correct / len(ref_vals)

In [ ]:
def financial_f1(pred, ref):

    p = financial_precision(pred, ref)
    r = financial_recall(pred, ref)

    if p + r == 0:
        return 0.0

    return 2 * p * r / (p + r)

In [ ]:
pred = """営業収益48兆367億円、税引前利益6兆415億円、当期純利益が4兆7650億円
営業利益が4兆7955億円

【トヨタの2025年3月期の業績】
・営業収益48兆367億円（前期比＋2兆9,413億円、＋6.5%）
・営業利益4兆7,955億円（前期比△5,573億円、△10.4%）
・税引前利益6兆4,145億円（前期比△5,504億円、△7.9%）
・当期利益4兆7,650億円（前期比△1,798億円、△3.6%）"""
ref = """営業収益48兆367億円、税引前利益6兆415億円、当期純利益が4兆7650億円
営業利益が4兆7955億円"""

numerical_accuracy(pred, ref)

{'pred_numbers': [48,
  367,
  6,
  415,
  4,
  7650,
  4,
  7955,
  2025,
  3,
  48,
  367,
  2,
  9413,
  6,
  5,
  4,
  7955,
  5573,
  10,
  4,
  6,
  4145,
  5504,
  7,
  9,
  4,
  7650,
  1798,
  3,
  6],
 'ref_numbers': [48, 367, 6, 415, 4, 7650, 4, 7955],
 'exact_match_rate': 1.0,
 'avg_relative_error': np.float64(0.0)}

In [ ]:
def llm_judge_evaluation(pipeline, query, context, answer):



    judge_prompt = f"""
あなたは金融レポート評価専用AIです。
感想は禁止。機械的に採点してください。

【質問】
{query}

【参考文書】
{context}

【AI回答】
{answer}

## 評価手順（必ず順番に実行）

STEP1:
参考文書に存在する数値をすべて抽出せよ。

STEP2:
AI回答に含まれる数値をすべて抽出せよ。

STEP3:
両者を比較し、
- 一致数
- 存在しない数値（幻覚）
を判定せよ。

STEP4:
以下で採点：

・文書根拠性 (0-3)
・質問適合性 (0-3)
・数値一致性 (0-4)

## 出力形式（厳守）

DocNumbers: [...]
AnswerNumbers: [...]
HallucinatedNumbers: [...]

Score: 数字(0-10)
Reason: 1文で説明
"""





    output = pipeline(
        [{"role": "user", "content": judge_prompt}],
        max_new_tokens=800,
        temperature=0.1
    )

    return output[0]["generated_text"][-1]["content"]

In [ ]:
from google.colab import files

pdf_file = "トヨタ_2025_03_有価証券報告書 (1).pdf"

analyzer = HybridAnalyzer(pdf_file)
analyzer.prepare()

1/4: 数値データを抽出中...
2/4: 全文テキストを読み込み中（50ページ）...
3/4: RAGデータベースを構築中...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


4/4: Llama-3.2-3B モデルをロード中...


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

--- 全準備完了 ---


In [ ]:
question = "2025年3月期の業績ハイライトと、その主な要因、今後の見通しを教えてください。"
result = analyzer.analyze(question)

search_results = analyzer.vectorstore.similarity_search(question, k=4)
context = "\n\n".join([d.page_content for d in search_results])

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [ ]:
judge_result = llm_judge_evaluation(
    analyzer.summarizer_pipeline,
    question,
    context,
    result
)

print("=== LLM Judge Result ===")
print(judge_result)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== LLM Judge Result ===
**評価結果**

**STEP1**
参考文書に存在する数値をすべて抽出します。
- 48兆367億円
- 2兆9,413億円
- 6.5%
- 4.7%
- 3兆9,402億円
- 6,811億円
- 10.4%
- 7.9%
- 3.6%
- 1,450億円
- 5,573億円
- 5,504億円
- 1,798億円

**STEP2**
AI回答に含まれる数値をすべて抽出します。
- 48兆367億円
- 4兆7,955億円
- 6兆4,145億円
- 4兆7,650億円
- 1,450億円

**STEP3**
両者を比較し、次の結果が得られます。
- 一致数: 5
- 存在しない数値（幻覚）：0

**STEP4**
採点結果は次の通りです。
- 文書根拠性: 3 (48兆367億円、4兆7,955億円、6兆4,145億円、4兆7,650億円)
- 質問適合性: 3 (トヨタの決算のデータ、業績要因、今後の見通し)
- 数値一致性: 4 (48兆367億円、4兆7,955億円、6兆4,145億円、4兆7,650億円)

**評価結果**
AI回答は参考文書と一致しています。AI回答はトヨタの決算のデータ、業績要因、今後の見通しについて正確に説明しています。ただし、数値一致性は 4 つだけです。
